# FZ0-Deep-Risk (Contribution 1) — Germany A1, clean run

**Before running:** replace `fz0_deep_risk.py` in the folder with the LATEST
version (composite epoch selection + `med_bal`), then restart the kernel.

Flow (run top to bottom):
1. Config + data
2. Smoke test (1 seed, verbose) — **check `med_bal` ≈ 0.50**
3. Coverage sanity check
4. Full multi-seed training (default hyperparameters)
5. Raw backtest + DM vs the two-stage champion
6. **ACI recalibration → the official A1 number** (expect coverage ≈ 0.95, Kupiec pass, FZ0 in the realistic range −3.8 to −4.6; anything beyond −5 = degeneracy, do not trust)
7. (Optional) Optuna — documented as unstable; skip for the frozen A1 verdict

## 1. Config + data

In [ ]:
import importlib, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from darts import TimeSeries
import pipeline_v2p as pv2p, var_v2p as vv, fz0_deep_risk as fz
importlib.reload(fz); importlib.reload(vv)

CC, TARGET, H, INPUT_LEN = "GER", "ger_bmk", 5, 90
DATASET = "dataset/ds_steel.xlsx"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

d = pv2p.preprocess(TARGET, DATASET)
w = fz.make_windows(d, input_len=INPUT_LEN, h=H)
sra = dict(zip(d["sra"].time_index, d["sra"].univariate_values()))

def to_price(vals):
    ts = TimeSeries.from_times_and_values(w["test_time"], vals.reshape(-1, 1), columns=[TARGET])
    return pv2p.inverse_chain(ts, d["scaler"]).univariate_values()

print("device:", DEVICE, "| n_cov:", w["n_cov"],
      "| train/val/test:", len(w["Xtr"]), len(w["Xva"]), len(w["Xte"]))

device: cuda | n_cov: 6 | train/val/test: 3158 751 1002


## 2. Smoke test (1 seed, verbose)

Watch the log: `val` should DECREASE without NaN/explosion, and **`med_bal` must
stay ≈ 0.50** (healthy median). If `med_bal` drifts to 0.1 or 0.9, stop and report.

In [10]:
torch.manual_seed(42); np.random.seed(42)
model = fz.FZ0DLinear(INPUT_LEN, n_cov=w["n_cov"])
model, val_best = fz.fit(model, w, epochs=60, warmup=15, patience=20, lr=1e-3,
                         device=DEVICE, verbose=True)
print("smoke val (composite):", round(val_best, 4))

  ep   0 | val=5.8305 (fz=4.386 pin=0.1445 med_bal=0.71) | best=5.8305
  ep  20 | val=-1.0517 (fz=-1.347 pin=0.0295 med_bal=0.50) | best=-1.1100
smoke val (composite): -1.11


## 3. Coverage sanity check (raw bands)

In [11]:
m, v, e = fz.predict(model, w, device=DEVICE)
yhat = to_price(m); q95 = to_price(m + v)
y = np.array([sra[t] for t in w["test_time"]])
L = np.log(y / yhat); VaR = np.log(q95 / yhat)
print(f"median balance P(L>0): {float((L>0).mean()):.3f}  (target ~0.50)")
print(f"raw empirical 95% coverage: {1-(L>VaR).mean():.3f}")
print(f"exceedances: {int((L>VaR).sum())} of {len(L)} (expected ~{0.05*len(L):.0f})")

median balance P(L>0): 0.454  (target ~0.50)
raw empirical 95% coverage: 0.951
exceedances: 49 of 1002 (expected ~50)


## 4. Full multi-seed training (default hyperparameters)

Defaults: `INPUT_LEN=90` (cell 1), `kernel=25` (class default), `lr=1e-3`.
Median seed by the composite validation loss.

In [12]:
runs = []
for s in [42, 43, 44]:
    torch.manual_seed(s); np.random.seed(s)
    mdl = fz.FZ0DLinear(INPUT_LEN, n_cov=w["n_cov"])
    mdl, vf = fz.fit(mdl, w, epochs=300, warmup=20, patience=25, lr=1e-3,
                     device=DEVICE, verbose=False)
    runs.append((s, vf, mdl)); print(f"seed {s}: val={vf:.4f}")
runs.sort(key=lambda r: r[1])
seed_med, val_med, model = runs[len(runs)//2]
print("median seed:", seed_med, "| val (composite):", round(val_med, 4))

seed 42: val=-1.1679
seed 43: val=-1.2132
seed 44: val=-1.2147
median seed: 43 | val (composite): -1.2132


## 5. Raw backtest + DM vs the two-stage champion (TCN / GJR-GARCH-t)

In [13]:
m, v, e = fz.predict(model, w, device=DEVICE)
yhat = to_price(m); q95 = to_price(m + v); es = to_price(m + e)
y = np.array([sra[t] for t in w["test_time"]])
L_e = np.log(y / yhat)
V_e = np.log(q95 / yhat)
E_e = np.maximum(np.log(es / yhat), V_e)          # ES >= VaR after transport

hits = (L_e > V_e).astype(int)
fz0_raw = float(np.nanmean(vv.fz0_loss(L_e, np.maximum(V_e, 1e-4), E_e, vv.A)))
print("=== FZ0-DLinear raw (end-to-end), Germany ===")
print(f"FZ0={fz0_raw:.4f} | coverage={1-hits.mean():.3f} | Kupiec={vv.kupiec(hits, vv.A):.3f} "
      f"| DQ_str={vv.dq_stride(hits, V_e, H):.3f} | exc={int(hits.sum())}")

# two-stage champion baseline (TCN residuals + GJR-GARCH-t)
RF = f"result/v2p_result_steel_{CC.lower()}.xlsx"
_, L_b, _ = vv.load_losses(RF, CC, H, "tcn")
V_b, E_b = vv.rolling_var_garch(L_b, vv.A, "GJR")
W = vv.WINDOW
fz_b = vv.fz0_loss(L_b[W:], np.maximum(V_b[W:], 1e-4), np.maximum(E_b[W:], V_b[W:]), vv.A)
fz_e = vv.fz0_loss(L_e, np.maximum(V_e, 1e-4), E_e, vv.A)
n = min(len(fz_b), len(fz_e))
stat, p = vv.dm_loss(fz_e[:n], fz_b[:n], nw_lag=H)
print(f"DM raw e2e vs TCN/GJR-GARCH: dFZ0={np.nanmean(fz_e[:n])-np.nanmean(fz_b[:n]):+.4f} | p={p:.3f}")

=== FZ0-DLinear raw (end-to-end), Germany ===
FZ0=-3.7923 | coverage=0.920 | Kupiec=0.000 | DQ_str=0.001 | exc=80
DM raw e2e vs TCN/GJR-GARCH: dFZ0=+0.8525 | p=0.999


## 6. ACI recalibration — the official A1 number

The framework's conformal stage applied to the e2e native VaR. Sanity gates:
coverage ≈ 0.95 with Kupiec pass, and **FZ0 within −3.8 to −4.6** (beyond −5 =
degeneracy: distrust and report).

In [14]:
V_aci, E_aci = vv.aci_qr(L_e, V_e)
mask = np.isfinite(V_aci[W:])
La = L_e[W:][mask]; Va = np.maximum(V_aci[W:][mask], 1e-4)
Ea = np.maximum(E_aci[W:][mask], Va)
hits_a = (La > Va).astype(int)
fz0_aci = float(np.nanmean(vv.fz0_loss(La, Va, Ea, vv.A)))
print("=== FZ0-DLinear + ACI (official A1), Germany ===")
print(f"FZ0={fz0_aci:.4f} | coverage={1-hits_a.mean():.3f} | "
      f"Kupiec={vv.kupiec(hits_a, vv.A):.3f} | DQ_str={vv.dq_stride(hits_a, Va, H):.3f} "
      f"| exc={int(hits_a.sum())} (expected ~{0.05*len(La):.0f})")

fz_e_aci = vv.fz0_loss(La, Va, Ea, vv.A)
n2 = min(len(fz_b), len(fz_e_aci))
stat2, p2 = vv.dm_loss(fz_e_aci[:n2], fz_b[:n2], nw_lag=H)
print(f"DM e2e+ACI vs TCN/GJR-GARCH: dFZ0={np.nanmean(fz_e_aci[:n2])-np.nanmean(fz_b[:n2]):+.4f} "
      f"| p={p2:.3f}  ({'e2e better' if p2 < 0.05 else 'not significant / two-stage better'})")

# diagnostics guard (from the -7.09 episode): distrust if bands collapse
print(f"[guard] V_aci p5/p50/p95: {np.round(np.percentile(Va, [5,50,95]), 5)} "
      f"| days V<=1e-3: {int((Va<=1e-3).sum())} | mean ln(e): {float(np.mean(np.log(Ea))):.2f}")

=== FZ0-DLinear + ACI (official A1), Germany ===
FZ0=-4.3115 | coverage=0.955 | Kupiec=0.552 | DQ_str=0.674 | exc=34 (expected ~38)
DM e2e+ACI vs TCN/GJR-GARCH: dFZ0=+0.2393 | p=0.903  (not significant / two-stage better)
[guard] V_aci p5/p50/p95: [0.00014 0.00907 0.02375] | days V<=1e-3: 52 | mean ln(e): -4.35


## 7. (Optional) Optuna — documented instability, skip for the frozen A1

FZ0-based hyperparameter tuning proved unstable out of sample (test coverage
oscillated 0.95 → 0.99 → 0.89 across configs, even with a validation-coverage
gate — regime shift GER val 2019-22 vs test 2023-26). Kept here for the record;
the frozen A1 verdict uses the DEFAULT configuration + ACI (cells 4-6).

In [15]:
RUN_OPTUNA = False   # set True only for stability experiments
if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        il  = trial.suggest_int("input_len", 60, 240, step=30)
        ker = trial.suggest_int("kernel", 5, 51, step=2)
        lr  = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        wloc = fz.make_windows(d, input_len=il, h=H)
        torch.manual_seed(42); np.random.seed(42)
        mdl = fz.FZ0DLinear(il, n_cov=wloc["n_cov"], kernel=ker)
        mdl, vf = fz.fit(mdl, wloc, epochs=200, warmup=30, patience=25, lr=lr,
                         device=DEVICE, verbose=False)
        with torch.no_grad():
            mv, vv_, _ = mdl(torch.tensor(wloc["Xva"], device=DEVICE),
                             torch.tensor(wloc["Cva"], device=DEVICE))
        cov = float(np.mean((wloc["Zva"] - mv.cpu().numpy()) <= vv_.cpu().numpy()))
        return vf + 50.0 * max(0.0, abs(cov - 0.95) - 0.02)
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50)
    print("best:", study.best_params, "| val:", round(study.best_value, 4))

In [16]:
import inspect, fz0_deep_risk as fz
src = inspect.getsource(fz)
print("seleção composta:", "val_pin * 10.0" in src)
print("med_bal:", "med_bal" in src)
print("m.detach no objetivo:", "z - m.detach()" in src)
print("init estreito:", "init_gap=-2.5" in src)

seleção composta: True
med_bal: True
m.detach no objetivo: True
init estreito: True
